# Task 1 ensemble

Every sub-model cell trains one model and stores it in `assignment1_2026/trained_models/task1/<model>.pt`.
The blend cells only need the stored files, so after a kernel restart nothing has to be retrained.
All hyperparameters: `lib/config.py`, Task 1 section.

In [1]:
%load_ext autoreload
%autoreload 2

# assignment1.py and lib/ live next to this notebook (in submission/)
import sys
from pathlib import Path
sys.path.append(str(Path.cwd()))

import torch

import assignment1
from lib.models.autorec import load_autorec, store_autorec
from lib.models.bayesian_factorization_machine import load_bayesian_factorization_machine, store_bayesian_factorization_machine
from lib.models.bias_model import load_bias_model, store_bias_model
from lib.models.blender import load_blender, store_blender
from lib.config import TASK1_LIGHTGBM_BLENDER, TASK1_RIDGE_BLENDER, TASK1_SUBMITTED_BLENDER
from lib.models.factorization_machine import load_factorization_machine, store_factorization_machine
from lib.models.integrated_model import load_integrated_model, store_integrated_model
from lib.models.item_knn import load_item_knn, store_item_knn
from lib.models.rating_transformer import load_rating_transformer, store_rating_transformer
from lib.models.SVDpp import load_svd, load_svdpp, store_svd, store_svdpp
from lib.models.zero_model import load_zero_model, store_zero_model

print(torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

2.14.1+cu130 | CUDA available: True | NVIDIA GeForce RTX 5080


### Data and validation
Ratings $\mathcal R = \{(u, i, r_{ui})\}$, $r_{ui} \in [0, 5]$, split at random into 90 % training $\mathcal T$ and 10 % validation $\mathcal V$ (`VALIDATION_FRACTION`, `SPLIT_SEED`).
Every sub-model: fit on $\mathcal T$ $\rightarrow$ predictions $\hat r^{(m)}_{ui}$ for $\mathcal V$ (input of the blender) $\rightarrow$ refit on $\mathcal T \cup \mathcal V$ $\rightarrow$ test predictions.

$$\mathrm{MSE} = \frac{1}{|\mathcal V|}\sum_{(u,i)\in\mathcal V}\big(r_{ui}-\hat r_{ui}\big)^2$$

In [2]:
# the test file has no timestamps -> the task has no time data at all: no Task 1 model can use time
task1 = assignment1.load_task1()

## Sub-models

### Bias model (PML Eq. 5.10 without factors)
$$\hat r_{ui} = \mu + b_u + b_i$$
$$\min_{b}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \lambda_{b_u}\sum_u b_u^2 + \lambda_{b_i}\sum_i b_i^2$$
Closed form, alternating for 10 rounds ($U_i$ = users who rated $i$, $I_u$ = items rated by $u$,
$\lambda_{b_u}$ = `user_bias_regularization`, $\lambda_{b_i}$ = `item_bias_regularization`):
$$b_i \leftarrow \frac{\sum_{u\in U_i}\left(r_{ui}-\mu-b_u\right)}{|U_i|+\lambda_{b_i}}, \qquad b_u \leftarrow \frac{\sum_{i\in I_u}\left(r_{ui}-\mu-b_i\right)}{|I_u|+\lambda_{b_u}}$$

In [3]:
# bias model: mu + b_u + b_i in closed form (seconds)
bias_model_result = assignment1.train_task1_bias_model(task1)
store_bias_model(bias_model_result, "task1")

[08:01:35] bias model | valid MSE 0.7627
[08:01:36] stored bias_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\bias_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/bias_model.pt')

### SVD / latent factor model (PML Eq. 5.10)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top \mathbf p_u$$
$$\min_{b,\mathbf p,\mathbf q}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \lambda_{b_u}\sum_u b_u^2 + \lambda_{b_i}\sum_i b_i^2 + \lambda_f\Big(\sum_u\lVert\mathbf p_u\rVert^2+\sum_i\lVert\mathbf q_i\rVert^2\Big)$$
(PML Eq. 5.12 with the bias weight $\lambda_1$ split into $\lambda_{b_u}$ and $\lambda_{b_i}$; $\lambda_f = \lambda_2$.)
Adam on mini-batches (each parameter's penalty is spread over the ratings that use it), biases start at the closed-form bias model,
early stopping on the validation MSE, then a refit on all ratings for the best number of epochs.

In [4]:
# SVD (a few minutes on the RTX 5080)
svd_result = assignment1.train_task1_svd(task1)
store_svd(svd_result, "task1")

[08:01:37] epoch   0 | train MSE 0.6986 | valid MSE 0.7628  (starting point)
[08:01:44] epoch   1 | train MSE 0.6973 | valid MSE 0.7625
[08:01:50] epoch   2 | train MSE 0.6905 | valid MSE 0.7610
[08:01:56] epoch   3 | train MSE 0.6725 | valid MSE 0.7572
[08:02:03] epoch   4 | train MSE 0.6474 | valid MSE 0.7536
[08:02:09] epoch   5 | train MSE 0.6186 | valid MSE 0.7509
[08:02:16] epoch   6 | train MSE 0.5883 | valid MSE 0.7493
[08:02:22] epoch   7 | train MSE 0.5583 | valid MSE 0.7486
[08:02:28] epoch   8 | train MSE 0.5296 | valid MSE 0.7487
[08:02:35] epoch   9 | train MSE 0.5030 | valid MSE 0.7494
[08:02:41] epoch  10 | train MSE 0.4786 | valid MSE 0.7504
[08:02:48] epoch  11 | train MSE 0.4566 | valid MSE 0.7517
[08:02:54] epoch  12 | train MSE 0.4369 | valid MSE 0.7532
[08:03:00] epoch  13 | train MSE 0.4194 | valid MSE 0.7548
[08:03:00] best valid MSE 0.7486 at epoch 7
[08:03:08] refit epoch   1 | train MSE 0.7060
[08:03:15] refit epoch   2 | train MSE 0.7035
[08:03:22] refit epo

WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/svd.pt')

### SVD++ (Koren 2008; implicit feedback as in FISM, PML Sec. 5.3.2)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top\Big(\mathbf p_u + |N(u)|^{-1/2}\sum_{j\in N(u)}\mathbf y_j\Big)$$
$N(u)$ = the items $u$ rated in the data the model trains on; $\mathbf y_j$ is regularised with $\lambda_f$ like $\mathbf q_j$. Training as for SVD.

In [5]:
# SVD++ (the slowest sub-model, ~7 min on the RTX 5080)
svdpp_result = assignment1.train_task1_svdpp(task1)
store_svdpp(svdpp_result, "task1")

[08:03:52] epoch   0 | train MSE 0.6986 | valid MSE 0.7629  (starting point)
[08:04:31] epoch   1 | train MSE 0.6630 | valid MSE 0.7550
[08:05:10] epoch   2 | train MSE 0.5671 | valid MSE 0.7452
[08:05:49] epoch   3 | train MSE 0.4480 | valid MSE 0.7431
[08:06:28] epoch   4 | train MSE 0.3494 | valid MSE 0.7470
[08:07:07] epoch   5 | train MSE 0.2772 | valid MSE 0.7531
[08:07:45] epoch   6 | train MSE 0.2270 | valid MSE 0.7590
[08:08:24] epoch   7 | train MSE 0.1926 | valid MSE 0.7646
[08:09:04] epoch   8 | train MSE 0.1688 | valid MSE 0.7697
[08:09:42] epoch   9 | train MSE 0.1527 | valid MSE 0.7740
[08:09:42] best valid MSE 0.7431 at epoch 3
[08:10:27] refit epoch   1 | train MSE 0.7031
[08:11:10] refit epoch   2 | train MSE 0.6553
[08:11:53] refit epoch   3 | train MSE 0.5565
[08:11:54] stored svdpp -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\svdpp.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/svdpp.pt')

### Item-item kNN on the bias model's residuals (PML Ch. 4; Koren 2010)
Baseline $b_{ui} = \mu + b_u + b_i$ (closed form), residuals $e_{uj} = r_{uj} - b_{uj}$.

Shrunk cosine similarity ($n_{ij}$ = users who rated both, $\lambda_s$ = `similarity_shrinkage`), only the $k$ most similar items $N_k(i)$ are kept:
$$s_{ij} = \frac{\sum_u e_{ui}\,e_{uj}}{\lVert\mathbf e_{\cdot i}\rVert\,\lVert\mathbf e_{\cdot j}\rVert}\cdot\frac{n_{ij}}{n_{ij}+\lambda_s}$$
$$\hat r_{ui} = b_{ui} + \frac{\sum_{j\in N_k(i)\cap I_u} s_{ij}\,e_{uj}}{\sum_{j\in N_k(i)\cap I_u} s_{ij} + \lambda_p}$$

In [6]:
# item-item kNN on the bias model's residuals (CPU, a few minutes)
item_knn_result = assignment1.train_task1_item_knn(task1)
store_item_knn(item_knn_result, "task1")

[08:12:14] item kNN | 5,095,554 neighbour links | valid MSE 0.7541
[08:12:36] stored item_knn -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\item_knn.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/item_knn.pt')

### Zero model: $P(r_{ui} = 0)$ (logistic regression, PML Sec. 3.3)
$$P(r_{ui}=0) = \sigma\big(w_0 + \mathbf w^\top \mathbf x_{ui}\big), \qquad \mathbf x_{ui} = \Big[\operatorname{logit}\tilde z_u,\ \operatorname{logit}\tilde z_i,\ \log(1+n_u),\ \log(1+n_i)\Big]$$
$$\tilde z_u = \frac{z_u + s_u\,\pi}{n_u + s_u}$$
$z_u$ = zeros of user $u$, $n_u$ = ratings of $u$ (likewise for item $i$ with $s_i$), $\pi$ = global share of zeros, $s$ = prior strength.
Training rows leave their own rating out; features standardised. The output is a probability, an input of the blender (not a rating).

In [7]:
# zero model: P(rating = 0) (CPU, under a minute)
zero_model_result = assignment1.train_task1_zero_model(task1)
store_zero_model(zero_model_result, "task1")

[08:12:36] zero model | valid AUC 0.8599 | log loss 0.0536 | mean P(0) 0.0149 vs share 0.0146
[08:12:37] stored zero_model -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\zero_model.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/zero_model.pt')

### Factorization machine (Rendle 2010; PML Eq. 6.3)
$$\hat r(\mathbf x) = \mu + \sum_f w_f x_f + \sum_{f<g}\langle\mathbf v_f,\mathbf v_g\rangle\,x_f x_g$$
One active feature per field: user $u$, item $i$, the user's activity bucket $a_u$ and the item's popularity bucket $a_i$ (deciles of #ratings):
$$\hat r_{ui} = \mu + w_u + w_i + w_{a_u} + w_{a_i} + \sum_{f<g\ \in\ \{u,\,i,\,a_u,\,a_i\}}\langle\mathbf v_f,\mathbf v_g\rangle$$
Trained like SVD (Adam, L2 penalty, early stopping, refit). No time information (Task 1 has none);
the time-aware version with calendar fields is part of Task 3.

In [8]:
# factorization machine (a few minutes on the RTX 5080)
# part of the blend (TASK1_BLENDED_MODELS in lib/config.py)
factorization_machine_result = assignment1.train_task1_factorization_machine(task1)
store_factorization_machine(factorization_machine_result, "task1")

[08:12:37] epoch   0 | train MSE 0.6996 | valid MSE 0.7647  (starting point)
[08:12:44] epoch   1 | train MSE 0.6959 | valid MSE 0.7632
[08:12:51] epoch   2 | train MSE 0.6886 | valid MSE 0.7610
[08:12:59] epoch   3 | train MSE 0.6770 | valid MSE 0.7588
[08:13:06] epoch   4 | train MSE 0.6577 | valid MSE 0.7566
[08:13:13] epoch   5 | train MSE 0.6312 | valid MSE 0.7553
[08:13:20] epoch   6 | train MSE 0.6005 | valid MSE 0.7564
[08:13:27] epoch   7 | train MSE 0.5692 | valid MSE 0.7594
[08:13:35] epoch   8 | train MSE 0.5389 | valid MSE 0.7640
[08:13:42] epoch   9 | train MSE 0.5111 | valid MSE 0.7696
[08:13:49] epoch  10 | train MSE 0.4857 | valid MSE 0.7754
[08:13:56] epoch  11 | train MSE 0.4626 | valid MSE 0.7811
[08:13:56] best valid MSE 0.7553 at epoch 5
[08:14:04] refit epoch   1 | train MSE 0.7063
[08:14:12] refit epoch   2 | train MSE 0.7024
[08:14:20] refit epoch   3 | train MSE 0.6949
[08:14:28] refit epoch   4 | train MSE 0.6825
[08:14:36] refit epoch   5 | train MSE 0.6636


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/factorization_machine.pt')

### Bayesian factorization machine (Freudenthaler et al. 2011; Rendle 2012, libFM; via myFM)
The factorization machine (PML Eq. 6.2) with four feature groups: user $u$, its implicit feedback $N(u)$, item $i$ and its implicit feedback $N(i)$ (the users who rated $i$):
$$\hat r_{ui} = \mu + w_u + w_i + \sum_{f<g}\langle\mathbf v_f,\mathbf v_g\rangle x_f x_g, \qquad x_j = |N(u)|^{-1/2}\ \text{for } j\in N(u), \quad x_v = |N(i)|^{-1/2}\ \text{for } v\in N(i)$$
Not one optimum: Gibbs sampling draws all parameters and the priors of every group from the posterior, the prediction is the mean over the kept draws
$$\hat r_{ui} = \frac{1}{S}\sum_{s=1}^{S}\hat r_{ui}\big(\Theta^{(s)}\big), \qquad \Theta^{(s)} \sim p(\Theta\mid\mathcal T)$$
No learning rate, no $\lambda$, no early stopping (the priors take the role of $\lambda$). Runs on the CPU (`pip install myfm`).

In [ ]:
# Bayesian factorization machine (Gibbs sampling, CPU: ~5 s per draw -> roughly 15-20 min per fit, the refit again)
bayesian_factorization_machine_result = assignment1.train_task1_bayesian_factorization_machine(task1)
store_bayesian_factorization_machine(bayesian_factorization_machine_result, "task1")

### Rating transformer (self-attention over the user's rated books; Vaswani et al. 2017)
History tokens of user $u$ (at most $L$ books, the asked book $i$ always hidden), plus one query token for $i$:
$$\mathbf t_j = \mathbf e_j + \mathbf e_{r_{uj}}\quad (j\in I_u\setminus\{i\}), \qquad \mathbf t_q = \mathbf e_i + \mathbf e_{\text{query}}$$
$$\mathbf h = \operatorname{Encoder}\big(\mathbf t_q, \mathbf t_{j_1}, \dots, \mathbf t_{j_L}\big)_q, \qquad \hat r_{ui} = \mu + b_u + b_i + \mathbf w^\top\mathbf h$$
The attention of $i$ to the user's books is a learned book-to-book similarity (like item kNN), the attended ratings the user representation (like SVD++).
No positions (Task 1 has no order). Biases start at the closed-form bias model, $\mathbf w$ at 0 $\rightarrow$ epoch 0 = bias model. Trained like SVD.

In [ ]:
# rating transformer (GPU)
rating_transformer_result = assignment1.train_task1_rating_transformer(task1)
store_rating_transformer(rating_transformer_result, "task1")

### I-AutoRec on the bias model's residuals (Sedhain et al. 2015)
Baseline $b_{ui} = \mu + b_u + b_i$ (closed form, fixed). Item-based autoencoder: book $i$ = the vector of its residuals $\mathbf x_i\in\mathbb R^{|U|}$, $x_{ui} = r_{ui} - b_{ui}$ (0 where a user did not rate it):
$$\mathbf h_i = \sigma(\mathbf V\mathbf x_i + \mathbf b), \qquad \hat r_{ui} = b_{ui} + \big(\mathbf W\mathbf h_i + \mathbf c\big)_u$$
$$\min_{\mathbf W,\mathbf V,\mathbf b,\mathbf c}\ \sum_{(u,i)\in\mathcal T}\big(r_{ui}-\hat r_{ui}\big)^2 + \frac{\lambda}{2}\Big(\lVert\mathbf W\rVert_F^2 + \lVert\mathbf V\rVert_F^2\Big)$$
Only observed ratings enter the loss; $\mathbf W, \mathbf c$ start at 0 $\rightarrow$ epoch 0 = bias model. The paper centres on $\mu$ only: on Task 1 that stayed far behind the bias model
(a book with few ratings has an almost empty $\mathbf x_i$). Non-linear $\rightarrow$ errors that differ from the factor models. Adam, early stopping, refit.

In [ ]:
# I-AutoRec (GPU)
autorec_result = assignment1.train_task1_autorec(task1)
store_autorec(autorec_result, "task1")

### Integrated model: SVD++ + learned neighbourhood (Koren 2008, Eq. 16)
$$\hat r_{ui} = \mu + b_u + b_i + \mathbf q_i^\top\Big(\mathbf p_u + |N(u)|^{-1/2}\sum_{j\in N(u)}\mathbf y_j\Big) + |R^k(i;u)|^{-1/2}\sum_{j\in R^k(i;u)}\big(r_{uj}-b_{uj}\big)\,w_{ij} + |N^k(i;u)|^{-1/2}\sum_{j\in N^k(i;u)} c_{ij}$$
$R^k(i;u)$ = the books $u$ rated among the $k$ neighbours of $i$ (shrunk residual cosine as in the item kNN), $N^k(i;u)$ = the books of $N(u)$ among them,
$b_{uj}$ = the fixed closed-form baseline. Unlike the item kNN, $w_{ij}$ and $c_{ij}$ are fitted to the ratings, together with the factors
(both start at 0, penalty $\lambda_n$). Trained like SVD++.

In [ ]:
# integrated model (GPU; the neighbour lookups run on the CPU once per batch)
integrated_model_result = assignment1.train_task1_integrated_model(task1)
store_integrated_model(integrated_model_result, "task1")

## Blend

### Blender input (stacking)
For every validation rating $(u,i)\in\mathcal V$: the predictions $\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui}$ of the sub-models in `blended_models`,
made by models that never saw $\mathcal V$. For the test rows: the predictions of the refitted sub-models.

In [9]:
# stored sub-models of the blend (also works right after a kernel restart)
sub_model_loaders = {
    "bias_model": load_bias_model,
    "svd": load_svd,
    "svdpp": load_svdpp,
    "item_knn": load_item_knn,
    "zero_model": load_zero_model,
    "factorization_machine": load_factorization_machine,
    "bayesian_factorization_machine": load_bayesian_factorization_machine,
    "rating_transformer": load_rating_transformer,
    "autorec": load_autorec,
    "integrated_model": load_integrated_model,
}
task1_model_results = {model_name: sub_model_loaders[model_name]("task1") for model_name in TASK1_SUBMITTED_BLENDER.blended_models}

### Ridge blend (linear stacking)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\theta_0 + \sum_{m=1}^{M}\theta_m\,\hat r^{(m)}_{ui}\Big), \qquad \min_{\boldsymbol\theta}\ \sum_{(u,i)\in\mathcal V}\big(r_{ui} - \theta_0 - \boldsymbol\theta^\top\hat{\mathbf r}_{ui}\big)^2 + \alpha\lVert\boldsymbol\theta\rVert^2$$
Score: 5-fold out-of-fold MSE on $\mathcal V$; the final blender is fitted on all of $\mathcal V$.

In [10]:
# ridge regression: learned weighted average of the sub-model predictions
ridge_blend_result = assignment1.train_task1_blender(task1, task1_model_results, TASK1_RIDGE_BLENDER)
store_blender(ridge_blend_result, "task1")

[08:14:37] bias_model             valid MSE 0.7627
[08:14:37] svd                    valid MSE 0.7486
[08:14:37] svdpp                  valid MSE 0.7431
[08:14:37] item_knn               valid MSE 0.7541
[08:14:37] factorization_machine  valid MSE 0.7553
[08:14:37] ridge blend                valid MSE 0.7296  (out-of-fold, 5 folds)
[08:14:37] ridge blend inputs: bias_model -3.24, svd -0.00236, svdpp 0.608, item_knn 3.39, zero_model 0.072, factorization_machine 0.236
[08:14:37] stored ridge_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\ridge_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/ridge_blender.pt')

### LightGBM blend (gradient-boosted regression trees)
$$\hat r_{ui} = \operatorname{clip}_{[0,5]}\Big(\sum_{t=1}^{T}\eta\,f_t(\mathbf x_{ui})\Big), \qquad \mathbf x_{ui} = \big[\hat r^{(1)}_{ui},\dots,\hat r^{(M)}_{ui},\ \log(1+n_u),\ \log(1+n_i)\big]$$
Each tree $f_t$ is fitted to the negative gradient of the squared error of the trees before it ($T$ = `lightgbm_n_estimators`, $\eta$ = learning rate).
Score: 5-fold out-of-fold MSE on $\mathcal V$.

In [11]:
# gradient-boosted trees (LightGBM) on the sub-model predictions + #ratings of user and item
lightgbm_blend_result = assignment1.train_task1_blender(task1, task1_model_results, TASK1_LIGHTGBM_BLENDER)
store_blender(lightgbm_blend_result, "task1")

[08:14:39] bias_model             valid MSE 0.7627
[08:14:39] svd                    valid MSE 0.7486
[08:14:39] svdpp                  valid MSE 0.7431
[08:14:39] item_knn               valid MSE 0.7541
[08:14:39] factorization_machine  valid MSE 0.7553
[08:14:39] lightgbm blend                valid MSE 0.7331  (out-of-fold, 5 folds)
[08:14:40] lightgbm blend inputs: bias_model 1.89e+03, svd 1.23e+03, svdpp 1.58e+03, item_knn 1.45e+03, zero_model 1.42e+03, factorization_machine 1.18e+03, log_user_ratings 1.65e+03, log_item_ratings 1.6e+03
[08:14:40] stored lightgbm_blender -> C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\trained_models\task1\lightgbm_blender.pt


WindowsPath('C:/Dateien/Workspace/UCSD/Recommender_Systems/recommender-systems/assignment1_2026/trained_models/task1/lightgbm_blender.pt')

### Submission
The test predictions $\hat r_{ui}$ of the blender chosen in `TASK<n>_SUBMITTED_BLENDER` go into the `rating` column.

In [12]:
# the submitted blend (TASK1_SUBMITTED_BLENDER in lib/config.py) writes output/task_1_predictions.csv
submitted_blend_result = load_blender("task1", TASK1_SUBMITTED_BLENDER.method)
assignment1.write_task1_predictions(submitted_blend_result.test_predictions)
print("wrote", assignment1.TASK1_PREDICTION_FILE)

wrote C:\Dateien\Workspace\UCSD\Recommender_Systems\recommender-systems\assignment1_2026\submission\output\task_1_predictions.csv
